# 🔥 FireWatch — FULL training on a Colab GPU (stable)

Trains the **TensorFlow / KerasHub RetinaNet** on the **full D-Fire dataset** on Colab's free
GPU. Uses a low, stabilized learning rate + early stopping so training converges instead of
diverging, and auto-saves the best model to your Drive.

### Before you start
1. **Runtime → Change runtime type → GPU → Save.**
2. Put the full **D-Fire** in the Drive of **the Google account you run Colab with** — either the
   `D-Fire` folder in My Drive, or a `D-Fire.zip` there (the data cell finds/unzips it).
3. Make sure the latest code is pushed to GitHub `master` (this notebook clones it).

Then **Runtime → Run all**.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU! Runtime > Change runtime type > GPU, then rerun.'

## 2. Get the code + install KerasHub
Clones `master`. If pip shows a **RESTART RUNTIME** button, click it, then re-run this cell.

In [ ]:
import os
if not os.path.isdir('fire-detection'):
    !git clone --depth 1 -b master https://github.com/01End/fire-detection.git
%cd fire-detection
!pip install -q -U keras-hub opencv-python-headless
import keras_hub, keras
print('keras', keras.__version__, '| keras_hub', keras_hub.__version__)

## 3. Mount Drive and locate the dataset (auto-detects layout)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

ZIP  = '/content/drive/MyDrive/D-Fire.zip'   # used only if there's no D-Fire folder
DEST = '/content/data'
FOLDER = '/content/drive/MyDrive/D-Fire'

search_base = FOLDER if os.path.isdir(FOLDER) else DEST
if search_base == DEST and not (os.path.isdir(DEST) and os.listdir(DEST)):
    assert os.path.exists(ZIP), f'No D-Fire folder and no zip at {ZIP}'
    !mkdir -p "{DEST}" && unzip -q -o "{ZIP}" -d "{DEST}"

root = layout = None
for dp, dns, fns in os.walk(search_base):
    if os.path.isdir(os.path.join(dp, 'train', 'images')):
        root, layout = dp, 'split_first'; break
    if os.path.isdir(os.path.join(dp, 'images', 'train')):
        root, layout = dp, 'type_first'; break
assert root, f'No train images found under {search_base}. Top: {os.listdir(search_base)}'

if layout == 'split_first':
    VAL_SPLIT = 'test' if os.path.isdir(f'{root}/test/images') else 'val'
    DATA = root
else:  # YOLO images/<split> -> symlink into <split>/images
    VAL_SPLIT = 'test' if os.path.isdir(f'{root}/images/test') else 'val'
    DATA = '/content/data_norm'
    for sp in ['train', VAL_SPLIT]:
        os.makedirs(f'{DATA}/{sp}', exist_ok=True)
        for kind in ['images', 'labels']:
            dst = f'{DATA}/{sp}/{kind}'
            if not os.path.exists(dst):
                os.symlink(f'{root}/{kind}/{sp}', dst)

IMG = 512
OUT = '/content/drive/MyDrive/fire_retinanet_full.weights.h5'
print('DATA =', DATA, '| VAL_SPLIT =', VAL_SPLIT, '| layout =', layout)
print('train images:', len(os.listdir(f'{DATA}/train/images')),
      f'| {VAL_SPLIT}:', len(os.listdir(f'{DATA}/{VAL_SPLIT}/images')))

## 4. Train (stable) 🚀
Low LR (1e-4) + reduce-LR-on-plateau + early-stopping (keeps the best weights), so it converges
instead of diverging. Best model auto-saves to Drive. Early stopping ends it when val stops
improving — `--epochs 20` is just a ceiling. ~26 min/epoch at 512px on a T4.

In [ ]:
!python -m training.tf_train \
    --data "{DATA}" \
    --train-split train \
    --val-split {VAL_SPLIT} \
    --epochs 20 \
    --batch-size 8 \
    --image-size {IMG} \
    --lr 0.0001 \
    --out "{OUT}"

## 5. Measure accuracy on the held-out test set 📊
Try score 0.5 first; raise/lower to trade precision vs recall.

In [ ]:
!python -m training.tf_eval \
    --model "{OUT}" \
    --data "{DATA}/{VAL_SPLIT}" \
    --image-size {IMG} --score-thr 0.5 --limit 500

## 6. See it work — annotated detections 🔥

In [ ]:
import glob, shutil
os.makedirs('val_sample', exist_ok=True)
picked = 0
for lf in sorted(glob.glob(f'{DATA}/{VAL_SPLIT}/labels/*.txt')):
    if os.path.getsize(lf) > 0:
        stem = os.path.splitext(os.path.basename(lf))[0]
        ip = f'{DATA}/{VAL_SPLIT}/images/' + stem + '.jpg'
        if os.path.exists(ip):
            shutil.copy(ip, 'val_sample/'); picked += 1
    if picked >= 8:
        break

!python -m firewatch.cli detect \
    --model "{OUT}" \
    --input val_sample \
    --backend tf --arch retinanet \
    --image-size {IMG} --score-thr 0.5 \
    --out detect_out

from IPython.display import Image, display
for p in sorted(glob.glob('detect_out/*'))[:8]:
    print(p); display(Image(p))